# BayMorph — Observed Single-Band Sérsic Fit

This notebook demonstrates the canonical BayMorph workflow for a
user-supplied astronomical FITS image.

BayMorph does not distribute observational data with the source repository or
installed package. Set the input path and target seed below for the observation
being analysed.

Two configuration routes are available:

- **guided** — the recommended evidence-aware workflow using
  `ObservationAdvisor` and observational flux-prior calibration;
- **direct** — an expert workflow using `make_config_from_fits` with explicitly
  declared scientific priors.

Both routes produce the same BayMorph configuration format and feed the same
production inference pipeline.

The Advisor may inspect the fitted image to report diagnostic numerical
scales. These target diagnostics are not used automatically to define
scientific prior bounds.

For total flux, the guided workflow can instead construct an observationally
calibrated prior from detectability and detector saturation. The measured
target flux is not used to define those bounds.


In [ ]:
from pathlib import Path

from baymorph import (
    BayMorphRunner,
    ObservationAdvisor,
    make_config_from_fits,
)


## 1. Observation and analysis paths

Set `fits_path` to the user-supplied science image. Relative paths are resolved
from the directory where Jupyter is running; an absolute path may also be used.

One analysis corresponds to one output directory. BayMorph writes generated
products only to the user-selected output location.


In [ ]:
analysis_name = "observed_galaxy_single_sersic"

fits_path = Path(
    "data/observed_galaxy.fits"
)
outputs_root = Path("runs")
analysis_dir = outputs_root / analysis_name
config_path = analysis_dir / "config.yaml"

# Use "center" only when the target is near the image centre.
# Otherwise replace it with an explicit zero-based (x, y) pixel coordinate.
target_position = "center"

print(f"FITS: {fits_path}")
print(f"Analysis directory: {analysis_dir}")


## 2. Inspect the observation

`ObservationAdvisor` is an optional evidence-aware inspection and
configuration layer.

Image-derived target quantities are reported as diagnostic numerical scales
and are not automatically converted into scientific prior bounds.

The guided flux calibration is different: it uses the declared morphology
priors together with the prepared footprint, noise model, PSF, and detector
saturation metadata to determine observationally relevant flux support. The
measured target flux is not used to centre or define the calibrated bounds.

Inspect the summary before continuing and verify that the target seed,
observational metadata, and detected source are appropriate.


In [ ]:
advisor = ObservationAdvisor(
    fits_path,
    target_position=target_position,
)

print(advisor.summary())


## 3. Build the configuration

The recommended `"guided"` workflow first creates an observational workspace,
prepares the image and empirical PSF, and then calibrates the total-flux prior.

The calibration uses one ensemble of morphology draws and exploits linear
flux scaling to determine:

- a lower bound from a declared detectability requirement;
- an upper bound from detector saturation.

The resulting prior is shown to the user and must be explicitly accepted
before the final scientific configuration is written.

This generic tutorial uses a normalized Gaussian likelihood. Select a
correlated likelihood only after validating an appropriate covariance model
for the supplied observation and its normalization requirements.

Set `configuration_mode` to `"direct"` only when the scientific priors,
including the total-flux prior, are being declared explicitly by the user.


In [ ]:
configuration_mode = "guided"

common_options = {
    "output_path": config_path,
    "object_name": analysis_name,
    "outputs_root": outputs_root,
    "psf_type": "empirical",
    "likelihood": {
        "type": "gaussian",
        "include_normalization": True,
    },
    "sampler": {
        "name": "nautilus",
        "execution": {
            "processes": 10,
            "progress": True,
        },
        "init": {
            "n_live": 300,
            "seed": 42,
        },
        "run": {
            "n_eff": 500,
            "f_live": 0.02,
        },
    },
}


if configuration_mode == "guided":
    flux_calibration = advisor.calibrate_flux_prior(
        prepare=True,
        build_psf=True,
        snr_threshold=5.0,
        required_detectable_fraction=0.95,
        required_unsaturated_fraction=0.95,
        n_draws=256,
        random_state=42,
        bootstrap_samples=512,
        confidence_level=0.95,
        model_name="single_sersic",
        **common_options,
    )

    print(flux_calibration.summary())

    config = advisor.configure(
        flux_calibration=flux_calibration,
        **common_options,
    )

    if config is None:
        raise RuntimeError(
            "Configuration cancelled."
        )

elif configuration_mode == "direct":
    direct_flux_prior = None

    if direct_flux_prior is None:
        raise ValueError(
            "Direct configuration requires an explicitly declared "
            "direct_flux_prior."
        )

    config = make_config_from_fits(
        fits_path,
        target_position=target_position,
        model_name="single_sersic",
        model_priors={
            "flux": direct_flux_prior,
        },
        **common_options,
    )

else:
    raise ValueError(
        "configuration_mode must be 'direct' or 'guided'."
    )


config


## 4. All-in-one workflow

This is the normal BayMorph user path.

`runner.run(diagnostics=True)` performs image preparation, empirical-PSF
construction, likelihood-specific preparation, Bayesian inference,
persistence, and configured diagnostics.


In [ ]:
runner = BayMorphRunner(
    config_path
)

result = runner.run(
    diagnostics=True
)

result


## 5. Step-by-step equivalent

The same production workflow can be executed explicitly when intermediate
products need to be inspected.

    runner = BayMorphRunner(config_path)

    runner.prepare_image()
    runner.select_psf_candidates()
    runner.build_psf()

    result = runner.fit()

    runner.diagnostics()

`runner.fit()` resolves likelihood-specific dependencies such as the fixed
correlated-noise calibration when required.

The all-in-one and step-by-step interfaces use the same configuration,
probabilistic model, and inference machinery.
